# Anomaly Detection in Azure ML Studio

### 1. Business Overview

This document provides a complete technical specification of the Anomaly Detection pipeline built in Microsoft Azure AI Machine Learning Studio. Its purpose is to give business executives, product owners, and governance stakeholders a plain-language understanding of every technical decision — from how data enters the system to how the model is evaluated and registered for production use.

The system addresses a clear business problem: identifying fraudulent credit card transactions within a dataset where genuine fraud is extremely rare — fewer than 2 in every 1,000 transactions. The pipeline is built on a Jupyter Notebook hosted within Azure AI Machine Learning, combining cloud-managed infrastructure with Python-based data science logic.

#### Business Problem Statement

Credit card fraud causes direct financial losses and damages customer trust. The challenge is not simply detecting fraud — it is detecting it accurately within a dataset where 99.83% of transactions are legitimate. A model that flags everything as normal would appear 99.8% accurate while catching zero fraud. This specification describes how we avoid that trap.



### 2. System Architecture & Infrastructure

The solution is built entirely within Microsoft Azure, using the following platform services:

| Azure Service | Role in This System | Business Purpose |
|--------------|--------------------|------------------|
| Azure Subscription & Workspace | Hosting environment for all ML resources | Provides governance, access control, and billing boundary |
| Azure AI Machine Learning Studio | Primary development and execution environment | Enables collaborative, auditable notebook development in the cloud |
| Azure ML Datasets (Assets > Data) | Registered home of the credit card fraud dataset | Single source of truth — prevents version drift across experiments |
| Azure ML Model Registry | Storage for the trained model artefact | Enables version-controlled deployment and rollback capability |
| Jupyter Notebook (.ipynb) | Executable specification and analysis document | Combines code, outputs, explanations, and charts in one reviewable file |
| Python 3.9 Runtime | Execution environment for all data processing and modelling | Standard, reproducible environment compatible with local and cloud execution |

### Fraud Detection Pipeline

The illustration depicts our Fraud detection pipeline.

 ![My Screenshot](fraud_detection_pipeline.png)

#### Portability Note

The notebook is designed to run in two environments without code changes: directly within Azure AI Machine Learning Studio (cloud-hosted), or locally on a developer machine with Python 3.9. This dual-mode design supports both rapid iteration by engineers and formal review runs in the governed cloud environment.


### 3. Data Pipeline

#### 3.1 Source Dataset

The dataset used is the Kaggle European Credit Card Fraud Detection dataset, which contains 284,807 transactions recorded over two days. It is pre-registered in Azure ML under the name creditcard_fraud within the workspace’s Assets > Data catalogue.

| Dataset Property | Value |
|------------------|-------|
| Total transactions | 284,807 |
| Fraudulent transactions | 492 (approximately 0.172%) |
| Legitimate transactions | 284,315 (approximately 99.83%) |
| Number of input features | 30 (Time, V1–V28, Amount) |
| Label column | Class (1 = fraud, 0 = normal) |

##

| Dataset Property | Value |
|------------------|-------|
| Storage location | Azure ML Workspace — Assets > Data > creditcard_fraud |
| Approximate file size | ~150 MB |


### 3.2  Data Loading

The notebook retrieves the registered dataset directly from Azure ML using the following pattern. This avoids re-uploading data for each experiment and ensures every team member is working from the same, version-controlled source.

This code initializes the analytical environment required to develop and deploy a machine learning model within the organization’s cloud analytics platform.

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


### Plain-Language Explanation

This code is the equivalent of a filing clerk retrieving a specific, version-stamped document folder from a managed archive room. Workspace.from_config() is the access key; Dataset.get_by_name() is the folder request; to_pandas_dataframe() opens the folder and spreads the contents on the desk for analysis.

In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

#### 3.3 Data Preparation & Normalisation

Before training, the data undergoes two preparation steps. These are standard practice in machine learning and are necessary for the algorithm to function correctly.

| Preparation Step | Business Rationale |
|------------------|--------------------|
| Standardise the Amount column | Transaction values range from £0.01 to £25,000+. Without scaling, the model’s decisions would be dominated by transaction size rather than behavioural patterns. Standardisation brings Amount onto the same numerical scale as all other features. |
| Remove the Time column | The Time feature records elapsed seconds from the first transaction in the batch. It does not represent calendar time and provides no meaningful signal for anomaly detection in this context. Removing it simplifies the model without loss of predictive power. |
| Separate features (X) from label (y) | Standard ML practice. X is the set of inputs the model uses to make predictions. y is the correct answer (fraud or not) used only to evaluate performance — never seen by the model during training. |
``

#### Why This Matters

Many machine learning algorithms — including Isolation Forest — perform better when numeric features are on a similar scale.  
Also, splitting the data into `X` and `y` is a standard step that prepares it for training and evaluation.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### 4. Machine Learning Model — Isolation Forest

#### 4.1  Algorithm Selection Rationale

The chosen algorithm is Isolation Forest, an unsupervised anomaly detection method developed specifically for high-dimensional, imbalanced datasets — precisely the profile of credit card fraud data.

The key insight behind Isolation Forest is counter-intuitive: rather than learning what fraud looks like, it learns what normal looks like and identifies everything that does not fit. It achieves this by randomly partitioning data using decision trees and measuring how quickly each transaction can be isolated from all others. Fraudulent transactions — being structurally different from the overwhelming majority — are isolated in far fewer steps.



In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### 4.2  Model Configuration

The model is initialised with two parameters that directly govern its sensitivity:

| Parameter | Value & Business Meaning |
|-----------|--------------------------|
| contamination | 0.0017 — Tells the model to expect approximately 0.17% of transactions to be anomalous, matching the known fraud rate in the dataset. This calibrates how aggressively the model flags outliers. |


## 5. Performance Evaluation & Metrics Framework

#### 5.1  Why Accuracy is the Wrong Metric

A naive model that classifies every single transaction as legitimate would achieve 99.83% accuracy — because 99.83% of transactions are legitimate. This is the fundamental trap of evaluating imbalanced classification problems using overall accuracy. It is misleading and should never be used as the headline metric in fraud detection contexts.


The following four metrics collectively provide an accurate picture of model performance for this use case:

| Metric       | What It Means                                                                 |
|--------------|--------------------------------------------------------------------------------|
| **Precision** | How often the model was *correct* when it said a transaction was fraud        |
| **Recall**    | How many of the *actual fraud cases* the model successfully found             |
| **F1-Score**  | A balance between precision and recall — like a combined performance score     |
| **Support**   | The number of examples in each group (normal or fraud) in the real data        |

#### Results Summary

The following results were produced by the initial Isolation Forest model configuration as described in this specification. These are baseline results that indicate the need for further model refinement:

#

| Class | Description | Precision / Recall / F1 | Support |
|------:|-------------|--------------------------|---------|
| 0 | Normal Transactions | 1.00 / 1.00 / 1.00 | 284,315 |
| 1 | Fraudulent Transactions | 0.29 / 0.28 / 0.28 | 492 |
| Overall Accuracy | — | 99.9% | 284,807 |

#### Performance Interpretation for Executives

The model is highly effective at confirming normal transactions, but currently catches fewer than 1 in 3 fraud cases. When it does flag a transaction as suspicious, it is only correct about 29% of the time. While the raw 99.9% accuracy figure sounds strong, it is almost entirely driven by the model’s performance on the 284,315 normal transactions. The metric that matters for business decision-making is the fraud-class F1 score of 0.28, which indicates substantial room for improvement through feature engineering, threshold tuning, and model retraining.



In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### 6. Model Registration & Versioning

Upon satisfactory evaluation, the trained model is serialised and registered in the Azure ML Model Registry. This step is optional during exploratory development but mandatory prior to any production deployment or governance review.

| Registration Property | Value & Purpose |
|-----------------------|-----------------|
| Serialisation format | Pickle (.pkl) via joblib — a compressed binary representation of the trained model object. This allows the model to be saved, transferred, and reloaded without retraining. |
| Registration name | creditcard_if_model — a human-readable identifier that groups all versions of this model within the Registry. |
| Azure ML Registry function | Model.register() uploads the model file and creates a versioned record in the workspace, accessible to deployment pipelines, auditors, and MLOps processes. |
| Versioning behaviour | Each call to Model.register() with the same name increments the version number automatically, providing a full audit trail of model changes over time. |

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Visualize a Count of Predicted Anomalies

### 7. Visualisation & Model Explainability

The pipeline produces three visualisations. Each serves a distinct governance or diagnostic purpose:

#### 7.1  Prediction Distribution Chart (Count Plot)
A bar chart showing the total number of transactions predicted as normal (0) versus anomalous (1). This is a rapid sanity check confirming that the model is not over- or under-flagging. A well-calibrated model should predict approximately 480–500 anomalies, consistent with the known fraud rate.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


#### 7.2  Transaction Amount by Prediction Class (Box Plot)
A box plot comparing the distribution of transaction amounts for predicted-normal versus predicted-fraud transactions. The chart reveals whether the model is making decisions based on transaction size alone — a potential bias risk. If predicted fraud exhibits a dramatically wider or higher amount range, feature engineering to reduce amount-driven bias should be considered.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


#### 7.3  SHAP Beeswarm Plot — Feature Importance
SHAP (SHapley Additive exPlanations) is an industry-standard technique for explaining the internal reasoning of machine learning models. For each transaction, SHAP calculates a numerical ‘contribution score’ for every input feature, showing how much each feature pushed the model toward or away from a fraud prediction.

| SHAP Chart Element | How to Read It |
|-------------------|----------------|
| Each dot | Represents a single transaction in the analysis sample |
| Each row | Represents one input feature (V1–V28, Amount) |
| Dot colour (Red = High, Blue = Low) | Indicates whether that feature’s value was high or low for that transaction |
| Horizontal position | Shows direction of influence: right = pushes toward fraud prediction; left = pushes toward normal prediction |
| Row ordering (top to bottom) | Features at the top are the most influential in the model’s decisions overall |
| Scatter spread | Wide scatter indicates the feature has variable impact; tight clusters indicate consistent influence |

#### Why This Matters: Regulatory & Audit Value of SHAP

In regulated financial environments, it is insufficient to simply know that the model flagged a transaction. Auditors, compliance officers, and regulators increasingly require that model decisions can be explained at the feature level. SHAP provides this evidence trail, allowing the team to demonstrate that the model is not using protected or inappropriate signals in its fraud detection decisions.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

### 8. Technical Dependencies

The following Python libraries are required. All are available via the PyPI package repository and are pre-installed in standard Azure ML compute environments:

| Package | Version Guidance | Primary Role | Azure ML Pre-installed |
|---------|------------------|--------------|------------------------|
| azureml-core | Latest stable | Workspace authentication, dataset access, model registration | Yes |
| pandas | >=1.3 | DataFrame manipulation and data exploration | Yes |
| scikit-learn | >=1.0 | Isolation Forest algorithm, classification metrics | Yes |
| matplotlib | >=3.4 | Charting and visualisation output | Yes |
| seaborn | >=0.11 | Statistical visualisation (count plot, box plot) | Yes |
| shap | >=0.40 | Model explainability and feature attribution | Yes |
| joblib | >=1.0 | Model serialisation for registry upload | Yes |

### 9. Authentication & Access Configuration

Connecting the notebook to the Azure ML Workspace requires authentication. Two approaches are supported, appropriate to different execution contexts:

| Authentication Method | When to Use & Configuration |
|-----------------------|-----------------------------|
| Azure CLI Login (az login) | Recommended for local development. The developer logs in via the Azure CLI and the SDK inherits credentials automatically. No config.json file is required. Note: a warning message (“Falling back to azure cli login credentials”) is expected and benign. |
| config.json file | Required when running within Azure AI Machine Learning Studio — Notebooks. The config.json file must be downloaded from the Azure portal (click the workspace name in the top-right corner) and uploaded to the same directory as the notebook. The notebook path variable must be updated to reference this file location. |
| Service Principal / MSI | Required for automated pipeline runs and production deployments. Not required for interactive notebook development. Recommended as the authentication standard for all non-human execution contexts. |

### 10. Known Limitations & Improvement Roadmap

| Limitation | Recommended Remediation |
|------------|--------------------------|
| Low fraud-class recall (28%) | Explore SMOTE oversampling, threshold tuning, or gradient-boosted ensemble models (XGBoost, LightGBM) with class-weight parameters calibrated for imbalanced datasets. |
| Low fraud-class precision (29%) | Review and suppress high-false-positive features using SHAP guidance. Apply Robust Scaling (IQR-based) and winsorisation to the Amount column to reduce outlier-driven false positives. |
| Accuracy as a misleading headline | Standardise AUPRC (Area Under Precision-Recall Curve) as the single headline KPI across all reporting. Prohibit accuracy-only reporting in model reviews and stakeholder dashboards. |
| SHAP limited to 100 rows | The current SHAP analysis is constrained to 100 transactions for performance. Expand to a full representative sample using background summarisation (shap.kmeans()) for production explainability reporting. |
| No train/test split applied | The current implementation trains and evaluates on the same data. Introduce a temporal 70/30 train/test split, or time-based holdout, to produce unbiased evaluation metrics. |
| Model evaluated once, not monitored | Implement Azure Monitor integration with automated data drift detection to trigger retraining when transaction pattern distributions shift significantly from the training baseline. |

### 11. Glossary of Technical Terms

The following terms appear throughout this specification. Definitions are written for a non-technical audience:

| Term | Plain-Language Definition |
|------|---------------------------|
| Algorithm | A step-by-step set of rules a computer follows to accomplish a task — in this case, to identify unusual transactions. |
| Anomaly Detection | The process of identifying transactions that behave differently from the established norm — a flag that something may be fraudulent. |
| AUPRC | Area Under the Precision-Recall Curve — a single score (0 to 1) that summarises how well the model balances catching fraud against avoiding false alarms. Higher is better; 1.0 is perfect. |
| Azure ML Workspace | A managed cloud environment in Microsoft Azure that houses all data, code, models, and compute resources for a machine learning project. |
| Contamination | A model parameter that tells the algorithm what fraction of the data to expect as anomalous. Must be set close to the known fraud rate for accurate calibration. |
| DataFrame | A tabular data structure — think of it as a spreadsheet in memory — used by the Python Pandas library to organise and manipulate data. |
| F1-Score | A combined performance score balancing Precision and Recall into a single number. Ranges from 0 (worst) to 1 (best). Preferred over accuracy for imbalanced datasets. |
| Feature | A measurable input variable used by the model to make predictions — for example, transaction amount, time of day, or a PCA-derived behavioural signal. |
| Isolation Forest | A machine learning algorithm that detects outliers by randomly partitioning data and measuring how quickly each data point can be separated from all others. Faster isolation = more anomalous. |
| Jupyter Notebook | An interactive document format that combines live executable code, results, charts, and written explanations in a single file. The standard format for cloud-based data science work. |
| Model Registry | A versioned catalogue within Azure ML that stores trained model artefacts, enabling controlled deployment, rollback, and audit tracking. |
| Normalisation / Standardisation | Rescaling numeric data so all features sit on a comparable scale. Prevents features with large numerical ranges (e.g., transaction amounts in pounds) from dominating features with small ranges. |
| PCA (Principal Component Analysis) | A mathematical technique that compresses many correlated variables into a smaller set of uncorrelated components. Features V1–V28 in this dataset are PCA-transformed for data privacy reasons. |
| Pickle / joblib | File formats used to save a trained Python model to disk, so it can be reloaded later without retraining. |
| Precision | Of all transactions flagged as fraud by the model, the proportion that were actually fraudulent. Low precision means many innocent customers are incorrectly flagged. |
| Random State | A fixed numerical seed that ensures a random process produces the same results every time. Essential for reproducible, auditable model training. |
| Recall | Of all truly fraudulent transactions in the dataset, the proportion that the model successfully identified. Low recall means fraudsters are slipping through undetected. |
| SHAP | SHapley Additive exPlanations — a technique that explains model predictions by quantifying how much each input feature contributed to a specific decision. |
| Workspace.from_config() | A Python function call that reads the Azure workspace connection file (config.json) and authenticates the notebook session against the Azure ML service. |
